# Structural Probe — Train & Save

Train a structural probe on toy sentences and save `probe_B.pt` for use in the demo notebook.

In [7]:
import torch, torch.nn as nn, numpy as np
from collections import deque
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW

MODEL_NAME, PROBE_RANK, LAYER = 'gpt2', 64, 8
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

Device: cuda


---
## Toy dataset

15 sentences with hand-written dependency heads (`0` = ROOT).

In [8]:
SENTENCES = [
    (['The', 'cat',    'sat',    'on',   'the',  'mat'],       [2,3,0,3,6,4]),
    (['Dogs','chase',  'cats',   'quickly'],                   [2,0,2,2]),
    (['She', 'saw',    'the',    'man',   'with', 'a','telescope'], [2,0,4,2,2,7,5]),
    (['The', 'chef',   'who',    'ran',   'to',  'the','store','is','out'], [2,8,4,2,4,7,5,0,8]),
    (['He',  'quickly','opened', 'the',   'door'],             [3,3,0,5,3]),
    (['The', 'bird',   'sang',   'a',     'sweet','song'],     [2,3,0,6,6,3]),
    (['John','gave',   'Mary',   'a',     'book'],             [2,0,2,5,2]),
    (['The', 'dog',    'that',   'barked','bit',  'the','man'],[2,5,4,2,0,7,5]),
    (['She', 'will',   'read',   'the',   'report'],          [3,3,0,5,3]),
    (['The', 'old',    'man',    'smiled','softly'],           [3,3,4,0,4]),
    (['The', 'boy',    'kicked', 'the',   'ball'],            [2,3,0,5,3]),
    (['Alice','loves', 'to',     'sing',  'loudly'],          [2,0,4,2,4]),
    (['The', 'small',  'cat',    'quietly','sleeps'],         [3,3,5,5,0]),
    (['We',  'saw',    'a',      'beautiful','sunset'],       [2,0,5,5,2]),
    (['The', 'teacher','praised','the',   'student'],         [2,3,0,5,3]),
]
print(f'{len(SENTENCES)} sentences')

15 sentences


---
## Helpers

In [9]:
def tree_distances(heads):
    N, adj = len(heads), [[] for _ in range(len(heads))]
    for i, h in enumerate(heads):
        if h > 0: adj[i].append(h-1); adj[h-1].append(i)
    dist = np.full((N, N), np.inf); np.fill_diagonal(dist, 0)
    for s in range(N):
        q, seen = deque([(s, 0)]), {s}
        while q:
            n, d = q.popleft(); dist[s, n] = d
            for nb in adj[n]:
                if nb not in seen: seen.add(nb); q.append((nb, d+1))
    return dist

def gold_edges(heads):
    return {frozenset([i, h-1]) for i, h in enumerate(heads) if h > 0}

---
## Extract hidden states

In [10]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
lm = AutoModel.from_pretrained(MODEL_NAME, output_hidden_states=True).to(DEVICE).eval()
N_LAYERS, HIDDEN_DIM = lm.config.num_hidden_layers, lm.config.hidden_size
print(f'{MODEL_NAME}: {N_LAYERS} layers, hidden_dim={HIDDEN_DIM}')

@torch.no_grad()
def get_word_states(words):
    enc = tokenizer(words, is_split_into_words=True, return_tensors='pt', add_special_tokens=True).to(DEVICE)
    w2t = {}
    [w2t.setdefault(w,[]).append(t) for t,w in enumerate(enc.word_ids()) if w is not None]
    layers = [torch.stack([hs[0].cpu()[w2t[i]].mean(0) for i in range(len(words))]) for hs in lm(**enc).hidden_states]
    return [layers[0]] + [l - layers[0] for l in layers[1:]]

# Pre-compute and cache
dataset = []
for words, heads in SENTENCES:
    dataset.append(dict(
        words      = words,
        heads      = heads,
        gold_dist  = torch.tensor(tree_distances(heads), dtype=torch.float32),
        gold_edges = gold_edges(heads),
        states     = get_word_states(words),
    ))
print(f'Cached hidden states for {len(dataset)} sentences.')

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

gpt2: 12 layers, hidden_dim=768
Cached hidden states for 15 sentences.


---
## Train structural probe

**Loss:** $\mathcal{L} = \frac{1}{|S|^2} \sum_{i,j} |d_{ij} - \|B(h_i - h_j)\|^2|$

In [11]:
class StructuralProbe(nn.Module):
    def __init__(self, hidden_dim, rank):
        super().__init__()
        self.B = nn.Parameter(torch.zeros(hidden_dim, rank))
        nn.init.uniform_(self.B, -0.05, 0.05)

    def forward(self, h):
        z = h @ self.B
        diff = z.unsqueeze(1) - z.unsqueeze(0)
        return (diff**2).sum(-1)

    def loss(self, h, gold):
        return (self(h) - gold).abs().sum() / h.shape[0]**2


probe = StructuralProbe(HIDDEN_DIM, PROBE_RANK).to(DEVICE)
opt   = AdamW(probe.parameters(), lr=1e-3)

for ep in range(1, 200):
    probe.train(); ep_loss = 0
    for s in dataset:
        h, g  = s['states'][LAYER].to(DEVICE), s['gold_dist'].to(DEVICE)
        loss  = probe.loss(h, g)
        opt.zero_grad(); loss.backward(); opt.step()
        ep_loss += loss.item()
    if ep % 50 == 0:
        print(f'Epoch {ep:3d}  loss={ep_loss/len(dataset):.4f}')

Epoch  50  loss=17.5071
Epoch 100  loss=191.1581
Epoch 150  loss=41.2476


---
## Save trained B

In [12]:
torch.save({
    'B':          probe.B.detach().cpu(),
    'layer':      LAYER,
    'hidden_dim': HIDDEN_DIM,
    'rank':       PROBE_RANK,
    'model':      MODEL_NAME,
}, 'probe_B.pt')
print('Saved probe_B.pt  — load in demo_uuas.ipynb')

Saved probe_B.pt  — load in demo_uuas.ipynb
